In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- 1. Load Data ---
print("Loading data...")
# Use the paths provided in the competition
TRAIN_PATH = '/kaggle/input/rnd-42-practice/BG_train_RG_2.csv'
TARGET_PATH = '/kaggle/input/rnd-42-practice/BG_train_RG_2_Result.csv'
TEST_PATH = '/kaggle/input/rnd-42-practice/BG_test_RG_2.csv'
SAMPLE_SUB_PATH = '/kaggle/input/rnd-42-practice/BG_sample_submission.csv'

train_feats = pd.read_csv(TRAIN_PATH)
train_targets = pd.read_csv(TARGET_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

# Merge targets into train features using ID
train_df = pd.merge(train_feats, train_targets, on='ID')

# --- 2. Feature Engineering ---
print("Engineering features...")

def process_data(df):
    # Convert Date to datetime
    if 'DT' in df.columns:
        df['DT'] = pd.to_datetime(df['DT'])
        # Extract Date components
        df['month'] = df['DT'].dt.month
        df['day'] = df['DT'].dt.day
        df['dayofweek'] = df['DT'].dt.dayofweek
        # Convert date to ordinal for trend learning
        df['date_int'] = df['DT'].map(pd.Timestamp.toordinal)
    
    # --- KEY STEP: Statistical Features on Lags ---
    # The columns N_D1_TRS .. N_D9_TRS represent past usage.
    # We aggregate them to capture user behavior patterns.
    lag_cols = [f'N_D{i}_TRS' for i in range(1, 10)]
    
    # Calculate row-wise statistics
    df['lag_mean'] = df[lag_cols].mean(axis=1)
    df['lag_std'] = df[lag_cols].std(axis=1)
    df['lag_min'] = df[lag_cols].min(axis=1)
    df['lag_max'] = df[lag_cols].max(axis=1)
    df['lag_sum'] = df[lag_cols].sum(axis=1)
    
    # Trend: Is recent usage (D1) higher than average past usage?
    df['trend_vs_mean'] = df['N_D1_TRS'] - df['lag_mean']
    
    return df

train_df = process_data(train_df)
test_df = process_data(test_df)

# --- 3. Encoding Categoricals ---
# Identify categorical columns (Prefix 'C_') and UID
cat_cols = [col for col in train_df.columns if col.startswith('C_')]

# We treat UID as categorical. If it has too many unique values, consider dropping it later.
if 'UID' in train_df.columns:
    cat_cols.append('UID')

encoder = LabelEncoder()
for col in cat_cols:
    # Combine train and test to ensure all categories are known
    # Convert to string to handle any mixed types
    combined = pd.concat([train_df[col], test_df[col]]).astype(str)
    encoder.fit(combined)
    train_df[col] = encoder.transform(train_df[col].astype(str))
    test_df[col] = encoder.transform(test_df[col].astype(str))

# --- 4. Validation Split (Time-Based) ---
# Sort by date to respect time in validation
train_df = train_df.sort_values('DT')

# Define features: All numeric columns except ID, Target Y, and original DT object
features = [c for c in train_df.columns if c not in ['ID', 'Y', 'DT']]
target = 'Y'

# Use the last 20% of data (timewise) for validation
split_idx = int(len(train_df) * 0.8)

X_train = train_df.iloc[:split_idx][features]
y_train = train_df.iloc[:split_idx][target]
X_val = train_df.iloc[split_idx:][features]
y_val = train_df.iloc[split_idx:][target]

print(f"Training on {len(X_train)} samples, Validating on {len(X_val)} samples.")

# --- 5. Model Training (LightGBM) ---
print("Training LightGBM...")

params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,        # Number of trees
    'learning_rate': 0.05,       # Step size
    'num_leaves': 31,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model = lgb.LGBMRegressor(**params)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='mae',
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
)

# Check Validation Score
val_preds = model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_preds)
print(f"\nValidation MAE: {val_mae:.4f}")

# --- 6. Final Prediction ---
print("Generating submission...")
test_preds = model.predict(test_df[features])

# Optional: Clip negative predictions if the target (usage) cannot be negative
# test_preds = np.maximum(test_preds, 0)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Y': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Saved to submission.csv. Ready to submit!")